In [1]:
import requests
import pandas as pd
import json

# Request download details from the pCloud link API
api_url = 'https://apilux3.pcloud.com/getpublinkdownload?fileid=93128819642&code=XZnqkRJZ98VUd3rEuMXAFIsEpAtnNHoKYK8k&linkpassword=&forcedownload=1'
response = requests.get(api_url)
response_data = response.json()

# Parse host and path to construct the final download URL
if 'hosts' in response_data and 'path' in response_data:
    download_host = response_data['hosts'][0]
    download_path = response_data['path']
    download_url = f'https://{download_host}{download_path}'
    print(f'Constructed Download URL: {download_url}')
else:
    raise ValueError('Could not retrieve valid host and path from API response.')

Constructed Download URL: https://plax1.pcloud.com/D4ZfVia1F7ZSysJQU7ZZZQbiBJkZ2ZZtjkZkZ8m1ZzLZ6pZeMZnqkRJZavncrCrEe70a8jHhgr6he5omJUgV/objet_consultations_1000.txt


In [ ]:
import requests

api_url = "https://apilux3.pcloud.com/getpublinkdownload?fileid=93128819642&code=XZnqkRJZ98VUd3rEuMXAFIsEpAtnNHoKYK8k&linkpassword=&forcedownload=1"
info = requests.get(api_url).json()
download_url = "https://" + info["hosts"][0] + info["path"]

r = requests.get(download_url)
with open("downloaded_file", "wb") as f:
    f.write(r.content)
print("Saved", len(r.content), "bytes")

Saved 112743 bytes


In [ ]:
# Load the plain text lines into a pandas DataFrame
with open('downloaded_file', 'r', encoding='utf-8', errors='ignore') as f:
    lines = [line.strip() for line in f if line.strip()]

df = pd.DataFrame(lines, columns=['objet_consultation'])
display(df.head())
print(f'Total rows loaded: {len(df)}')

,objet_consultation
0,"RD2E7; Travaux de reconstruction, de réparatio..."
1,Construction et installation de deux ombrières...
2,Marché Public de Maîtrise d'Oeuvre - Programme...
3,Location de deux balayeuses avec maintenance e...
4,Mission d'Assistance à Maitrise d'Ouvrage pour...


Total rows loaded: 998


In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
line_embeddings = model.encode(df["objet_consultation"].tolist(), show_progress_bar=True)
print(line_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(998, 1024)


In [ ]:
import numpy as np

np.savetxt("vectors.tsv", line_embeddings, delimiter="\t")
df["objet_consultation"].to_csv("metadata.tsv", index=False, header=False, sep="\t")
print("done")

done


In [ ]:
from sklearn.cluster import KMeans

df["objet_consultation"] = (
    df["objet_consultation"].str.replace(r"<br\s*/?>", " ", regex=True).str.strip()
)

K = 100
kmeans = KMeans(n_clusters=K, random_state=0, n_init="auto").fit(line_embeddings)
df["cluster"] = kmeans.labels_

# show 5 titles from 3 different clusters
for c in [0, 1, 2]:
    print(f"--- cluster {c} ---")
    for t in df[df["cluster"] == c]["objet_consultation"].head(5):
        print(t[:100])

--- cluster 0 ---
Sécurisation du virage de Mulsanne Circuit des 24 Heures du Mans Commune de Mulsanne Lot n°1 : Voiri
ACCORD-CADRE DE MAITRISE D'OEUVRE A MARCHES SUBSEQUENTS MARCHE SUBSEQUENT N°4 (SAINT BARTHELEMY D'AN
Marché subséquent dans le cadre de l'accord Cadre C2007:  Mission d'assistance à maîtrise d'ouvrage 
MS3 Accord-cadre mono-attributaire de maîtrise d'œuvre relatif à l’opération de réhabilitation du gr
ACCORD-CADRE DE MAITRISE D’ŒUVRE À MARCHÉS SUBSÉQUENTS Travaux de réfection d’étanchéité de toiture 
--- cluster 1 ---
Interventions de psychologues pour le Département de l'Hérault
Interventions de psychologues pour le Département de l'Hérault dans le cadre des missions des maison
REQUALIFICATION D'ESPACES PUBLICS ET EXTENSION DE LA MAIRIE DE PUY SANIERES
SCHEMA DE GESTION DES EAUX PLUVIALES DE LA COMMUNAUTE D'AGGLOMERATION FECAMP CAUX LITTORAL
CONSTRUCTION D'UNE STRUCTURE CMP/CATTP DE PEDOPSYCHIATRIE SUR LA COMMUNE DE PERTUIS - RELANCE DES LO
--- cluster 2 ---
Marché de